# QNN Integration: Individual Result-Set Plots

This notebook discovers result directories produced by `ExperimentResultWriter` and plots each run independently. Epoch curves are averaged across subsets and include a 95% confidence band when more than one subset is available.

In [ ]:
from pathlib import Path
import json
import re

from IPython.display import Markdown, display
import numpy as np
import pandas as pd
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots


pio.templates["latex"] = pio.templates["plotly_white"].update(
    layout=dict(
        colorway=px.colors.qualitative.D3,
        font=dict(
            family="CMU Serif",
            size=12
        )
    )
)

pio.templates.default = "latex"

pd.options.display.max_columns = 120
pd.options.display.max_colwidth = 160

In [ ]:
EPOCH_COLUMN_MAP = {
    "TrLoss": ("train", "loss"),
    "TrAcc": ("train", "accuracy"),
    "TrPrec": ("train", "precision"),
    "TrRec": ("train", "recall"),
    "TrF1": ("train", "f1"),
    "VaLoss": ("val", "loss"),
    "VaAcc": ("val", "accuracy"),
    "VaPrec": ("val", "precision"),
    "VaRec": ("val", "recall"),
    "VaF1": ("val", "f1"),
}
METRIC_ORDER = ["loss", "accuracy", "precision", "recall", "f1"]
SUBSET_RE = re.compile(r"subset_(\d+)_(epochs|test_metrics)\.csv$")


def locate_results_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = []
    for base in (cwd, *cwd.parents):
        candidates.extend([
            base / "experiments" / "QNN_integration" / "results",
            base / "results",
        ])

    for candidate in candidates:
        if candidate.exists() and any(candidate.glob("*/*/manifest.json")):
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not find experiments/QNN_integration/results from the current working directory."
    )


def subset_index(path: Path) -> int:
    match = SUBSET_RE.match(path.name)
    if not match:
        raise ValueError(f"Unexpected subset result filename: {path.name}")
    return int(match.group(1))


def discover_runs(results_root: Path) -> pd.DataFrame:
    records = []
    for manifest_path in sorted(results_root.glob("*/*/manifest.json")):
        run_dir = manifest_path.parent
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            manifest = {}

        pipeline = manifest.get("pipeline_name") or run_dir.parent.name
        timestamp = run_dir.name
        epoch_files = sorted(run_dir.glob("subset_*_epochs.csv"))
        metric_files = sorted(run_dir.glob("subset_*_test_metrics.csv"))
        records.append(
            {
                "run_id": f"{pipeline}/{timestamp}",
                "result_set": f"{pipeline} / {timestamp}",
                "pipeline": pipeline,
                "timestamp": timestamp,
                "run_dir": str(run_dir),
                "manifest_path": str(manifest_path),
                "started_at": manifest.get("started_at"),
                "completed_at": manifest.get("completed_at"),
                "subset_count_manifest": len(manifest.get("subsets", [])),
                "epoch_file_count": len(epoch_files),
                "test_metric_file_count": len(metric_files),
                "has_summary": (run_dir / "final_summary_across.csv").exists(),
            }
        )

    runs = pd.DataFrame.from_records(records)
    if runs.empty:
        return runs
    return runs.sort_values(["pipeline", "timestamp"]).reset_index(drop=True)


def load_epoch_history(runs: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for run in runs.itertuples(index=False):
        run_dir = Path(run.run_dir)
        for path in sorted(run_dir.glob("subset_*_epochs.csv")):
            df = pd.read_csv(path, sep=";")
            df.columns = [column.strip() for column in df.columns]
            for column in df.columns:
                df[column] = pd.to_numeric(df[column], errors="coerce")
            df["subset"] = subset_index(path)
            df["pipeline"] = run.pipeline
            df["timestamp"] = run.timestamp
            df["run_id"] = run.run_id
            df["result_set"] = run.result_set
            df["source_file"] = str(path)
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def load_test_metrics(runs: pd.DataFrame) -> pd.DataFrame:
    records = []
    for run in runs.itertuples(index=False):
        run_dir = Path(run.run_dir)
        for path in sorted(run_dir.glob("subset_*_test_metrics.csv")):
            metrics = pd.read_csv(path, sep=";")
            metric_row = {
                "subset": subset_index(path),
                "pipeline": run.pipeline,
                "timestamp": run.timestamp,
                "run_id": run.run_id,
                "result_set": run.result_set,
                "source_file": str(path),
            }
            for item in metrics.itertuples(index=False):
                metric_row[str(item.metric)] = pd.to_numeric(item.value, errors="coerce")
            records.append(metric_row)

    return pd.DataFrame.from_records(records)


def load_final_summaries(runs: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for run in runs.itertuples(index=False):
        path = Path(run.run_dir) / "final_summary_across.csv"
        if not path.exists():
            continue
        df = pd.read_csv(path, sep=";")
        for column in ["mean", "std"]:
            df[column] = pd.to_numeric(df[column], errors="coerce")
        df["pipeline"] = run.pipeline
        df["timestamp"] = run.timestamp
        df["run_id"] = run.run_id
        df["result_set"] = run.result_set
        df["source_file"] = str(path)
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def to_long_epoch_frame(epoch_history: pd.DataFrame) -> pd.DataFrame:
    value_columns = [column for column in EPOCH_COLUMN_MAP if column in epoch_history.columns]
    id_columns = [
        "Epoch",
        "subset",
        "pipeline",
        "timestamp",
        "run_id",
        "result_set",
        "source_file",
    ]
    long_df = epoch_history.melt(
        id_vars=id_columns,
        value_vars=value_columns,
        var_name="history_column",
        value_name="value",
    ).dropna(subset=["value"])

    column_meta = pd.DataFrame(
        [
            {"history_column": column, "split": split, "metric": metric}
            for column, (split, metric) in EPOCH_COLUMN_MAP.items()
        ]
    )
    return long_df.merge(column_meta, on="history_column", how="left")


def to_long_test_frame(test_metrics: pd.DataFrame) -> pd.DataFrame:
    value_columns = [column for column in ["loss", "accuracy", "precision", "recall", "f1"] if column in test_metrics.columns]
    id_columns = [
        "subset",
        "pipeline",
        "timestamp",
        "run_id",
        "result_set",
        "source_file",
    ]
    return test_metrics.melt(
        id_vars=id_columns,
        value_vars=value_columns,
        var_name="metric",
        value_name="value",
    ).dropna(subset=["value"])


def aggregate_with_ci(df: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    agg = (
        df.groupby(group_columns, dropna=False)["value"]
        .agg(mean="mean", std="std", count="count")
        .reset_index()
    )
    agg["std"] = agg["std"].fillna(0.0)
    agg["sem"] = np.where(agg["count"] > 0, agg["std"] / np.sqrt(agg["count"]), 0.0)
    agg["t_critical"] = agg["count"].apply(lambda n: stats.t.ppf(0.975, n - 1) if n > 1 else 0.0)
    agg["ci95"] = agg["sem"] * agg["t_critical"]
    agg["lower"] = agg["mean"] - agg["ci95"]
    agg["upper"] = agg["mean"] + agg["ci95"]
    return agg


def add_ci_band(fig: go.Figure, x, lower, upper, color: str, name: str, row=None, col=None) -> None:
    fig.add_trace(
        go.Scatter(
            x=list(x) + list(x)[::-1],
            y=list(upper) + list(lower)[::-1],
            fill="toself",
            fillcolor=color,
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip",
            name=name,
            showlegend=False,
        ),
        row=row,
        col=col,
    )


RESULTS_ROOT = locate_results_root()
runs = discover_runs(RESULTS_ROOT)
runs_with_epochs = runs[runs["epoch_file_count"] > 0].copy()
epoch_history = load_epoch_history(runs_with_epochs)
epoch_long = to_long_epoch_frame(epoch_history) if not epoch_history.empty else pd.DataFrame()
test_metrics = load_test_metrics(runs_with_epochs)
test_long = to_long_test_frame(test_metrics) if not test_metrics.empty else pd.DataFrame()
final_summaries = load_final_summaries(runs_with_epochs)

print(f"Results root: {RESULTS_ROOT}")
print(f"Discovered {len(runs)} manifest-backed result sets, {len(runs_with_epochs)} with epoch histories.")

In [ ]:
runs_with_epochs[[
    "result_set",
    "started_at",
    "completed_at",
    "epoch_file_count",
    "test_metric_file_count",
    "has_summary",
    "run_dir",
]]

## Select Result Sets

By default all discovered result sets with epoch histories are plotted. Edit `SELECTED_RESULT_SETS` to a smaller list when you want a shorter report.

In [ ]:
ALL_RESULT_SETS = runs_with_epochs["result_set"].tolist()
SELECTED_RESULT_SETS = ALL_RESULT_SETS

# Example:
# SELECTED_RESULT_SETS = ["Baseline_D / 30-04-2026-01-44"]

selected_result_sets = [name for name in SELECTED_RESULT_SETS if name in ALL_RESULT_SETS]
selected_result_sets

## Epoch Curves Per Result Set

In [ ]:
def plot_run_epoch_curves(result_set: str, include_subset_lines: bool = True) -> go.Figure:
    run_df = epoch_long[epoch_long["result_set"] == result_set].copy()
    if run_df.empty:
        raise ValueError(f"No epoch data found for {result_set!r}.")

    metrics = [metric for metric in METRIC_ORDER if metric in run_df["metric"].unique()]
    fig = make_subplots(
        rows=len(metrics),
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.045,
        subplot_titles=[metric.title() for metric in metrics],
    )

    split_styles = {
        "train": {"color": "#1f77b4", "dash": "solid", "band": "rgba(31,119,180,0.14)"},
        "val": {"color": "#ff7f0e", "dash": "dash", "band": "rgba(255,127,14,0.14)"},
    }

    for row_number, metric in enumerate(metrics, start=1):
        metric_df = run_df[run_df["metric"] == metric]

        if include_subset_lines:
            for (split, subset), subset_df in metric_df.groupby(["split", "subset"]):
                style = split_styles.get(split, {"color": "#999999", "dash": "dot"})
                subset_df = subset_df.sort_values("Epoch")
                fig.add_trace(
                    go.Scatter(
                        x=subset_df["Epoch"],
                        y=subset_df["value"],
                        mode="lines",
                        line=dict(color=style["color"], dash=style["dash"], width=1),
                        opacity=0.18,
                        name=f"{split} subset {subset}",
                        legendgroup=f"{metric}-{split}-subset",
                        showlegend=False,
                        hovertemplate=(
                            "subset=%{customdata[0]}<br>epoch=%{x}<br>value=%{y:.4f}<extra></extra>"
                        ),
                        customdata=subset_df[["subset"]],
                    ),
                    row=row_number,
                    col=1,
                )

        aggregated = aggregate_with_ci(metric_df, ["Epoch", "split", "metric"])
        for split, split_df in aggregated.groupby("split"):
            split_df = split_df.sort_values("Epoch")
            style = split_styles.get(split, {"color": "#444444", "dash": "solid", "band": "rgba(68,68,68,0.12)"})
            add_ci_band(
                fig,
                split_df["Epoch"],
                split_df["lower"],
                split_df["upper"],
                style["band"],
                f"{split} 95% CI",
                row=row_number,
                col=1,
            )
            fig.add_trace(
                go.Scatter(
                    x=split_df["Epoch"],
                    y=split_df["mean"],
                    mode="lines+markers",
                    line=dict(color=style["color"], dash=style["dash"], width=2.5),
                    marker=dict(size=5),
                    name=split,
                    legendgroup=split,
                    showlegend=(row_number == 1),
                    hovertemplate="epoch=%{x}<br>mean=%{y:.4f}<extra></extra>",
                ),
                row=row_number,
                col=1,
            )

    fig.update_layout(
        title=f"Epoch Metrics: {result_set}",
        height=max(360, 235 * len(metrics)),
        hovermode="x unified",
        legend_title_text="Split",
    )
    fig.update_xaxes(title_text="Epoch", row=len(metrics), col=1)
    fig.update_yaxes(title_text="Value")
    return fig


for result_set in selected_result_sets:
    display(Markdown(f"### {result_set}"))
    plot_run_epoch_curves(result_set).show()

## Test Metrics Per Result Set

In [ ]:
def plot_run_test_metrics(result_set: str) -> go.Figure:
    run_df = test_long[test_long["result_set"] == result_set].copy()
    if run_df.empty:
        raise ValueError(f"No test metrics found for {result_set!r}.")

    metric_order = [metric for metric in METRIC_ORDER if metric in run_df["metric"].unique()]
    fig = px.box(
        run_df,
        x="metric",
        y="value",
        color="metric",
        points="all",
        category_orders={"metric": metric_order},
        hover_data=["subset"],
        title=f"Test Metrics Across Subsets: {result_set}",
    )
    fig.update_layout(height=430, showlegend=False, xaxis_title="Metric", yaxis_title="Value")
    return fig


for result_set in selected_result_sets:
    if result_set in test_long["result_set"].unique():
        display(Markdown(f"### {result_set}"))
        plot_run_test_metrics(result_set).show()

## Final Summary Tables

In [ ]:
if final_summaries.empty:
    display(Markdown("No `final_summary_across.csv` files were found for the selected result sets."))
else:
    final_summaries[final_summaries["result_set"].isin(selected_result_sets)].sort_values(
        ["result_set", "metric"]
    )